# Economy of Bits — Notebook 03: final panel, and the bit-intensity index

Two sections.

**A (cells 3–5) settles the panel.** Notebook 02's automatic alias detector was wrong —
101 of 175 recovered firm-years came from false aliases like `CELG → NOW` and `ALXN → MRNA`,
because "EDGAR records a former name" is true of almost every long-lived filer. It is
replaced here by a curated table of 19 renames, each carrying its EDGAR evidence. Re-running
the JRFM estimators on the cleanly repaired panel is the robustness check that goes to the
editor.

**B (cells 6–10) is the companion paper's actual measurement.** Bit intensity built on the
coverage notebook 01 found, validated three ways, then the first tests of H1 and H2.

Upload `bits_inputs_v3.zip` — it carries notebook 01's EDGAR facts so nothing is
re-downloaded except ~19 recovered tickers. Runtime about 10 minutes.

## 1 · Setup

In [ ]:
# =============================================================================
# CELL 1 - Setup
#
# Notebook 02 established two things.
#
# ONE: the rename defect is real but IMMATERIAL. Re-estimating on a panel with
# only the verified renames applied gives alpha 1.467 -> 1.141 against a
# published 1.491 -> 1.141, and a Gini of 0.588 -> 0.719 against 0.586 -> 0.719.
# The JRFM paper's numbers survive. Section A below reproduces that cleanly so
# the result can go to the editor as a robustness check.
#
# TWO: notebook 02's automatic alias detector was wrong, and it is worth being
# precise about why. The rule was "EDGAR records a former name, and the filer's
# history predates the swap". Nearly every long-lived filer has a formerNames
# entry -- Skyworks, Molina and Moderna all do -- so the test passed for almost
# everyone and produced CELG -> NOW, ALXN -> MRNA, FRC -> AXON, INFO -> MOH.
# 101 of 175 recovered firm-years came from false aliases. They moved the
# numbers more than the true ones did, which is exactly why this was checked.
#
# The replacement is a curated table with per-firm evidence. No detector.
# =============================================================================
import os, re, sys, json, time, glob, zipfile, pathlib, datetime, shutil, warnings
import numpy as np, pandas as pd, requests

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 150)

try:
    import yfinance as yf
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
    import yfinance as yf

RUN_TAG = "bits_03_index"
BASE = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
ROOT = BASE / f"economy_of_bits_{RUN_TAG}"
OUTD, CACHE = ROOT / "outputs", ROOT / "cache"
FIGS = OUTD / "figures"
for d in (OUTD, CACHE, FIGS):
    d.mkdir(parents=True, exist_ok=True)
saved_tables = []

def savetable(df, name):
    if df is None or not len(df):
        print(f"  skipped {name} (empty)"); return
    df.to_parquet(OUTD / f"{name}.parquet"); df.to_csv(OUTD / f"{name}.csv", index=False)
    saved_tables.append(name); print(f"  saved {name}  ({len(df):,} rows)")

def cached(name, builder, refresh=False):
    p = CACHE / f"{name}.parquet"
    if p.exists() and not refresh:
        d = pd.read_parquet(p); print(f"  cache hit: {name} ({len(d):,} rows)"); return d
    d = builder()
    if d is not None and len(d):
        d.to_parquet(p); print(f"  cached {name} ({len(d):,} rows)")
    return d

PAPER_YEARS = list(range(2016, 2026))
HILL_FRAC, MIN_TAIL_K, MIN_N_ALPHA, MIN_N_INEQ, GROUP_MIN_K = 0.10, 20, 50, 30, 10
print(f"writing to: {ROOT}")

## 2 · Inputs

In [ ]:
# =============================================================================
# CELL 2 - Inputs
#
# Upload bits_inputs_v3.zip. It adds notebook 01's resolved EDGAR facts (43,126
# firm-year-field observations) so this notebook does not re-download them.
# =============================================================================
def find_inputs():
    for z in sorted(glob.glob(str(BASE / "**" / "bits_inputs_v3.zip"), recursive=True)):
        dest = CACHE / "inputs"; dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print(f"  extracted {pathlib.Path(z).name}"); return dest
    for d in (BASE / "inputs3", BASE, CACHE / "inputs"):
        if (pathlib.Path(d) / "edgar_facts_resolved.csv.gz").exists():
            return pathlib.Path(d)
    return None

SRC = find_inputs()
if SRC is None:
    raise FileNotFoundError("bits_inputs_v3.zip not found. Upload it to the file pane.")

panel = pd.read_csv(SRC / "sp500_marketcap_panel_clean.csv.gz")
panel["year"] = panel.year.astype(int)
est = panel[panel.year.isin(PAPER_YEARS)].copy()
rosters_long = pd.read_csv(SRC / "sp500_membership_reconstructed.csv")
ROSTER = {int(y): set(g.ticker) for y, g in rosters_long.groupby("year")
          if int(y) in PAPER_YEARS}
facts = pd.read_csv(SRC / "edgar_facts_resolved.csv.gz")
tick2cik = pd.read_csv(SRC / "ticker_to_cik.csv")
pub_ineq = pd.read_csv(SRC / "inequality_by_year.csv")
pub_alpha = pd.read_csv(SRC / "pareto_alpha_by_year.csv")
pub_group = pd.read_csv(SRC / "pareto_alpha_by_group.csv")
prereg = pd.read_csv(SRC / "platform_coding_preregistered.csv")

print(f"\npanel        {len(est):,} firm-years, {est.ticker.nunique()} tickers")
print(f"EDGAR facts  {len(facts):,} observations, {facts.cik.nunique()} CIKs, "
      f"{facts.field.nunique()} fields")
print(f"pre-registered platform coding: {len(prereg)} firms")
in_panel = {y: set(est[est.year == y].ticker) for y in PAPER_YEARS}

## 3 · The curated alias table

Nineteen entries with per-firm EDGAR evidence. No detector.

In [ ]:
# =============================================================================
# CELL 3 - The curated alias table
#
# Nineteen entries, each a corporate rename or ticker change verified against
# EDGAR's formerNames record with the date it took effect. The `evidence`
# column is the EDGAR entry, so a referee can check every line without running
# anything.
#
# The discipline that failed in notebook 02 and is enforced here: an alias is
# accepted only when EDGAR records a name change for the SAME filer, or the
# ticker change is documented and the CIK is unchanged. "The firm has some
# former name and has been filing a while" is not evidence, because it is true
# of almost every filer in the index.
# =============================================================================
ALIAS_TABLE = [
    # old,    new,     effective,     evidence from EDGAR formerNames / filings
    ("FB",    "META",  "2022-06-09", "Facebook Inc -> Meta Platforms, name to 2021-10-27, CIK 1326801"),
    ("ANTM",  "ELV",   "2022-06-28", "Anthem, Inc. -> Elevance Health, name to 2022-06-24, CIK 1156039"),
    ("DISCA", "WBD",   "2022-04-11", "Discovery -> Warner Bros. Discovery, CIK 1437107"),
    ("BLL",   "BALL",  "2022-05-10", "Ball Corp, ticker change only, CIK 9389"),
    ("ABC",   "COR",   "2023-08-30", "AmerisourceBergen -> Cencora, CIK 1140859"),
    ("RE",    "EG",    "2023-07-11", "Everest Re Group -> Everest Group, name to 2023-07-06, CIK 1095073"),
    ("PKI",   "RVTY",  "2023-05-18", "PerkinElmer -> Revvity, name to 2023-04-14, CIK 31791"),
    ("RVTY (PREVIOUSLY PKI)", "RVTY", "2024-01-01", "membership file writes the rename inline"),
    ("PEAK",  "DOC",   "2024-03-04", "Healthpeak Properties, ticker change only, CIK 765880"),
    ("STI",   "TFC",   "2019-12-09", "SunTrust + BB&T -> Truist Financial, CIK 92230"),
    ("FISV",  "FI",    "2023-06-08", "Fiserv, ticker change only, CIK 798354"),
    ("COG",   "CTRA",  "2021-10-04", "Cabot Oil & Gas -> Coterra Energy, CIK 858470"),
    ("CTL",   "LUMN",  "2020-09-18", "CenturyLink -> Lumen Technologies, name to 2021-01-20, CIK 18926"),
    ("TMK",   "GL",    "2019-08-08", "Torchmark -> Globe Life, name to 2019-08-02, CIK 320335"),
    ("HRS",   "LHX",   "2019-06-01", "Harris Corp -> L3Harris, name to 2019-06-25, CIK 202058"),
    ("NLOK",  "GEN",   "2022-11-08", "NortonLifeLock -> Gen Digital, name to 2022-10-12, CIK 849399"),
    ("WLTW",  "WTW",   "2022-01-20", "Willis Towers Watson, ticker change only, CIK 1140536"),
    ("JEC",   "J",     "2019-12-10", "Jacobs Engineering -> Jacobs Solutions, CIK 52988"),
    ("BRK.B", "BRK-B", "2023-09-24", "membership file punctuation, not a rename"),
]
aliases = pd.DataFrame(ALIAS_TABLE, columns=["old", "new", "effective", "evidence"])
ALIASES = dict(zip(aliases.old, aliases.new))
print(f"{len(ALIASES)} curated aliases:")
display(aliases)

# The guard that notebook 02 lacked: an alias may not join two tickers that were
# in the index at the same time, whatever the other evidence says.
bad = []
for o, n in list(ALIASES.items()):
    both = [y for y in PAPER_YEARS
            if o in ROSTER.get(y, set()) and n in ROSTER.get(y, set())]
    if both:
        bad.append((o, n, both)); del ALIASES[o]
print(f"\ncollision guard: {len(bad)} removed" + (f" -- {bad}" if bad else ""))

REV = {v: k for k, v in ALIASES.items()}
gaps = []
for y in PAPER_YEARS:
    for t in sorted(ROSTER[y] - in_panel[y]):
        gaps.append({"year": y, "roster_ticker": t,
                     "current_ticker": ALIASES.get(t, t),
                     "recoverable": t in ALIASES})
gaps = pd.DataFrame(gaps)
RECOVER = sorted(gaps.loc[gaps.recoverable, "current_ticker"].unique())
print(f"\n{len(RECOVER)} tickers to download: {RECOVER}")
print(f"{int((~gaps.recoverable).sum())} roster entries stay missing -- acquired or")
print("delisted firms, which is the honest survivorship residual the paper reports.")
savetable(aliases, "alias_table_curated")
savetable(gaps, "coverage_gaps")

## 4 · Rebuild the panel

In [ ]:
# =============================================================================
# CELL 4 - Rebuild and re-estimate: the definitive panel
#
# Same split reconciliation as the paper: Yahoo's Close is adjusted to today's
# share basis, get_shares_full returns the contemporaneous count, so
#   M_it = P_it * S_it * prod(splits strictly after 31 Dec of year t).
# =============================================================================
def dl_prices(tk):
    fr = []
    px = yf.download(tk, start="2014-01-01", end="2026-01-01", progress=False,
                     group_by="ticker", auto_adjust=False, threads=True)
    for t in tk:
        try:
            s = px[t]["Close"] if isinstance(px.columns, pd.MultiIndex) else px["Close"]
            s = s.dropna()
            if len(s):
                fr.append(s.rename(t))
        except Exception:
            pass
    return pd.concat(fr, axis=1) if fr else pd.DataFrame()

def dl_shares(tk):
    rows = []
    for t in tk:
        try:
            s = yf.Ticker(t).get_shares_full(start="2014-01-01")
            if s is not None and len(s):
                s = s[~s.index.duplicated(keep="last")]
                rows += [{"ticker": t, "date": pd.Timestamp(d).tz_localize(None),
                          "shares": float(v)} for d, v in s.items()]
        except Exception:
            pass
        time.sleep(0.05)
    return pd.DataFrame(rows, columns=["ticker", "date", "shares"])

def dl_splits(tk):
    rows = []
    for t in tk:
        try:
            s = yf.Ticker(t).splits
            if s is not None and len(s):
                rows += [{"ticker": t, "date": pd.Timestamp(d).tz_localize(None),
                          "ratio": float(r)} for d, r in s.items() if r and float(r) > 0]
        except Exception:
            pass
        time.sleep(0.03)
    return pd.DataFrame(rows, columns=["ticker", "date", "ratio"])

px_new = cached("rec_prices", lambda: dl_prices(RECOVER))
sh_new = cached("rec_shares", lambda: dl_shares(RECOVER))
sp_new = cached("rec_splits", lambda: dl_splits(RECOVER))
print(f"prices for {sum(t in px_new.columns for t in RECOVER)} of {len(RECOVER)}")

ypx = px_new.copy()
if len(ypx):
    ypx.index = pd.to_datetime(ypx.index)
    try:
        ypx.index = ypx.index.tz_localize(None)
    except (TypeError, AttributeError):
        ypx.index = ypx.index.tz_convert(None)
    ypx = ypx.resample("YE").last()
if len(sh_new):
    sh_new["year"] = pd.to_datetime(sh_new.date).dt.year
    ysh = (sh_new.sort_values("date").groupby(["ticker", "year"]).shares.last()
           .unstack(0).reindex(range(2014, 2027)).ffill().bfill())
else:
    ysh = pd.DataFrame()

def sfac(t, y):
    if not len(sp_new):
        return 1.0
    s = sp_new[(sp_new.ticker == t) & (sp_new.date > pd.Timestamp(f"{y}-12-31"))]
    return float(s.ratio.prod()) if len(s) else 1.0

recs = []
for y in PAPER_YEARS:
    rows = ypx.loc[ypx.index.year == y] if len(ypx) else pd.DataFrame()
    if not len(rows):
        continue
    prow = rows.iloc[-1]
    for t in sorted(ROSTER[y] - in_panel[y]):
        cur = ALIASES.get(t)
        if cur is None or cur not in prow.index or cur not in getattr(ysh, "columns", []):
            continue
        p_, s_ = prow.get(cur), (ysh.at[y, cur] if y in ysh.index else np.nan)
        if pd.isna(p_) or pd.isna(s_) or p_ <= 0 or s_ <= 0:
            continue
        f = sfac(cur, y)
        recs.append({"year": y, "ticker": cur, "roster_ticker": t, "price": float(p_),
                     "shares_adj": float(s_) * f, "mktcap": float(p_) * float(s_) * f,
                     "recovered": True})
recovered = pd.DataFrame(recs)
print(f"{len(recovered):,} recovered firm-years, "
      f"{recovered.ticker.nunique() if len(recovered) else 0} firms")

TRUTH_BN = {("META", 2021): 935, ("META", 2019): 585, ("META", 2017): 512,
            ("ELV", 2021): 113, ("WBD", 2022): 23, ("TFC", 2020): 64,
            ("LUMN", 2020): 11, ("GL", 2019): 11, ("LHX", 2020): 37}
chk = []
for (t, y), tb in TRUTH_BN.items():
    d = recovered[(recovered.ticker == t) & (recovered.year == y)] if len(recovered) else []
    if len(d):
        got = float(d.mktcap.iloc[0]) / 1e9
        chk.append({"ticker": t, "year": y, "recovered_bn": round(got),
                    "reference_bn": tb, "ratio": round(got / tb, 3)})
check = pd.DataFrame(chk)
if len(check):
    display(check)
    worst = float((check.ratio - 1).abs().max())
    print(f"largest deviation {100*worst:.1f}%  "
          f"{'OK' if worst < 0.20 else 'STOP -- recovery is wrong'}")
    assert worst < 0.20, "recovered market caps fail validation"
    savetable(check, "recovery_validation")

base = est.copy(); base["recovered"] = False; base["roster_ticker"] = base.ticker
cols = ["year", "ticker", "roster_ticker", "price", "shares_adj", "mktcap",
        "sector", "sub_industry", "recovered"]
for c in cols:
    if c not in base.columns:
        base[c] = np.nan
    if len(recovered) and c not in recovered.columns:
        recovered[c] = np.nan
rep = (pd.concat([base[cols], recovered[cols]], ignore_index=True) if len(recovered)
       else base[cols].copy())
rep = rep.sort_values(["year", "ticker", "recovered"]).drop_duplicates(
    ["year", "ticker"], keep="first")

# carry sector across a rename, then within a ticker
rep["sector"] = rep.sector.replace(["nan", "None", ""], np.nan)
known = rep.dropna(subset=["sector"]).drop_duplicates("ticker").set_index("ticker").sector
rep["sector"] = rep.sector.fillna(rep.ticker.map(known)).fillna(rep.roster_ticker.map(known))
print(f"\nfinal panel: {len(rep):,} firm-years, {rep.ticker.nunique()} firms, "
      f"{int(rep.recovered.sum())} recovered")
savetable(rep, "panel_final")

## 5 · Re-estimate — the table for the editor

In [ ]:
# =============================================================================
# CELL 5 - Re-estimate, and the table that goes to the editor
# =============================================================================
import statsmodels.api as sm
from scipy import stats as st

def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    if n == 0 or x.sum() == 0:
        return np.nan
    return float((2 * (np.arange(1, n + 1) * x).sum()) / (n * x.sum()) - (n + 1) / n)

def top_share(x, q):
    x = np.sort(np.asarray(x, float))[::-1]
    if not len(x) or x.sum() == 0:
        return np.nan
    return float(x[:max(1, int(round(q * len(x))))].sum() / x.sum())

def hhi(x):
    s = np.asarray(x, float)
    if not len(s) or s.sum() == 0:
        return np.nan
    s = s / s.sum(); return float((s ** 2).sum())

def hill_alpha(x, k=None):
    x = np.sort(np.asarray(x, float))[::-1]; x = x[x > 0]
    if k is None:
        k = max(10, int(HILL_FRAC * len(x)))
    k = min(k, len(x) - 1)
    if k < 5:
        return np.nan, np.nan, k
    a = 1.0 / np.mean(np.log(x[:k]) - np.log(x[k]))
    return float(a), float(a / np.sqrt(k)), int(k)

def measures(p, value="mktcap"):
    rows = []
    for y, g in p.groupby("year"):
        x = g[value].dropna().values; x = x[x > 0]
        if len(x) < MIN_N_INEQ:
            continue
        rec = {"year": int(y), "n": len(x), "gini": gini(x), "hhi": hhi(x),
               "top1_share": top_share(x, .01), "top10_share": top_share(x, .10)}
        if len(x) >= MIN_N_ALPHA:
            a, se, k = hill_alpha(x, int(HILL_FRAC * len(x)))
            rec.update({"alpha_hill": a, "alpha_hill_se": se, "hill_k": k})
        rows.append(rec)
    return pd.DataFrame(rows).sort_values("year").reset_index(drop=True)

def hac(d, c, maxlags=3):
    d = d[["year", c]].dropna().sort_values("year")
    X = sm.add_constant(d.year - d.year.mean())
    f = sm.OLS(d[c], X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    o = sm.OLS(d[c], X).fit()
    return (float(d[c].iloc[0]), float(d[c].iloc[-1]), float(f.params.iloc[1]),
            float(f.pvalues.iloc[1]), float(o.pvalues.iloc[1]))

final = measures(rep)
savetable(final, "inequality_alpha_final")

pub = pub_ineq.merge(pub_alpha[["year", "alpha_hill"]], on="year", how="left")
PUB_TREND = {"alpha_hill": 0.006, "gini": 0.0001, "top1_share": 0.0001,
             "top10_share": 0.0001, "hhi": 0.0001}
rows = []
for c in ["alpha_hill", "gini", "top1_share", "top10_share", "hhi"]:
    if c not in final.columns or c not in pub.columns:
        continue
    f0, f1, b, ph, po = hac(final, c)
    p0, p1 = float(pub[c].iloc[0]), float(pub[c].iloc[-1])
    rows.append({"measure": c, "published_2016": round(p0, 4), "published_2025": round(p1, 4),
                 "published_p": PUB_TREND[c],
                 "final_2016": round(f0, 4), "final_2025": round(f1, 4),
                 "final_slope": round(b, 5), "final_p_hac": round(ph, 4),
                 "final_p_ols": round(po, 4),
                 "shift_2016": round(f0 - p0, 4), "shift_2025": round(f1 - p1, 4),
                 "verdict": "HOLDS" if (np.sign(f1 - f0) == np.sign(p1 - p0)
                                        and ph < 0.05) else "CHECK"})
editor = pd.DataFrame(rows)
print("THE ROBUSTNESS TABLE FOR THE EDITOR\n")
display(editor)
savetable(editor, "editor_robustness_table")

cov = []
for y in PAPER_YEARS:
    cov.append({"year": y, "roster": len(ROSTER[y]),
                "published_n": int(pub.n[pub.year == y].iloc[0]),
                "final_n": int((rep.year == y).sum())})
covdf = pd.DataFrame(cov)
covdf["pct_published"] = (100 * covdf.published_n / covdf.roster).round(1)
covdf["pct_final"] = (100 * covdf.final_n / covdf.roster).round(1)
print("\ncoverage:")
display(covdf)
savetable(covdf, "coverage_final")

print("\n" + "=" * 70)
if (editor.verdict == "HOLDS").all():
    print("Every published claim survives the repair. The paper stands, and this")
    print("table is a robustness check worth sending the editor unprompted:")
    print("coverage in 2016 rises from "
          f"{covdf.pct_published.iloc[0]:.0f}% to {covdf.pct_final.iloc[0]:.0f}% "
          "of the roster and the conclusions do not move.")
else:
    print("At least one claim moved. Stop and send me this table before writing.")
print("=" * 70)

## 6 · Bit-intensity components

Gross margin cannot be required: banks have no cost of goods sold. The index is built on
what exists, and R&D blanks stay blank rather than becoming zeros.

In [ ]:
# =============================================================================
# CELL 6 - Bit-intensity components
#
# SECTION B begins. The companion paper's actual measurement.
#
# Notebook 01 settled what is available: revenue 97%, assets 99%, PP&E 84%,
# goodwill or intangibles 92%, gross profit 63%, R&D 38%. Gross margin cannot be
# required -- banks and insurers do not report cost of goods sold because they
# have no goods, and that is a quarter of the index. So the index is built on
# what exists and gross margin becomes a robustness check on the subsample where
# it is defined.
#
# Four components, each a direct reading of one claim in Machine, Platform,
# Crowd chapter 6:
#
#   asset lightness   1 - PP&E/assets        no physical distribution constraint
#   intangible share  (goodwill+intangibles)/assets   value carried in bits
#   R&D intensity     R&D/revenue            output produced by design, not plant
#   gross margin      (rev-COGS)/rev         marginal cost of reproduction ~ 0
#
# R&D BLANKS ARE NOT ZEROS. A firm that does no R&D reports nothing rather than
# zero, so a blank is ambiguous between "spent nothing" and "did not report".
# Filling zeros would collapse every non-R&D firm onto one point and manufacture
# the separation H1 is meant to test. Blanks stay blank and the index averages
# over whatever components a firm has, with a minimum of two.
# =============================================================================
# Yahoo's sector vocabulary leaked into the panel through notebook 02's
# fallback, so the same sector appears twice under two names -- "Healthcare"
# beside "Health Care", "Consumer Cyclical" beside "Consumer Discretionary".
# Left alone it splits sectors in every table and, worse, hides firms from the
# GICS digital flag.
YF_TO_GICS = {"Technology": "Information Technology",
              "Healthcare": "Health Care",
              "Financial Services": "Financials",
              "Consumer Cyclical": "Consumer Discretionary",
              "Consumer Defensive": "Consumer Staples",
              "Basic Materials": "Materials",
              "Industrials": "Industrials", "Energy": "Energy",
              "Utilities": "Utilities", "Real Estate": "Real Estate",
              "Communication Services": "Communication Services"}
rep["sector"] = rep.sector.replace(YF_TO_GICS)
print("sector vocabulary after harmonising:")
print(rep.sector.value_counts(dropna=False).to_string())

wide = facts.pivot_table(index=["cik", "year"], columns="field",
                         values="value", aggfunc="first").reset_index()
wide.columns.name = None
fund = wide.merge(tick2cik, on="cik", how="left")
F = rep.merge(fund, on=["ticker", "year"], how="left")
print(f"panel {len(rep):,} rows; matched to fundamentals "
      f"{int(F.revenue.notna().sum()):,} ({100*F.revenue.notna().mean():.1f}%)")

F["gross_profit"] = F.gross_profit.fillna(F.revenue - F.cogs)
F["asset_light"] = 1 - (F.ppe_net / F.assets)
# Intangibles EXCLUDING goodwill. Goodwill is the premium paid over book value
# in an acquisition, so goodwill/assets measures how acquisitive a firm has
# been, not how much of its value is carried in bits. A serial acquirer in
# industrials scores high on it. A dry run confirmed the problem: with goodwill
# included this component loaded NEGATIVELY on the first principal component,
# pulling against the other three. Goodwill is kept as a separate control.
F["intangible_share"] = F.intangibles / F.assets
F["goodwill_share"] = F.goodwill / F.assets
F["rnd_intensity"] = F.rnd / F.revenue
F["gross_margin"] = F.gross_profit / F.revenue

COMPONENTS = ["asset_light", "intangible_share", "rnd_intensity", "gross_margin"]
CONTROLS = ["goodwill_share"]
# Ratios built from accounting identities occasionally blow up on a tiny
# denominator. Winsorise at the 1st and 99th percentile WITHIN YEAR rather than
# dropping, so an extreme but real firm keeps its rank.
for c in COMPONENTS + CONTROLS:
    lo = F.groupby("year")[c].transform(lambda s: s.quantile(0.01))
    hi = F.groupby("year")[c].transform(lambda s: s.quantile(0.99))
    F[c] = F[c].clip(lo, hi)

print("\ncomponent coverage on the final panel:")
for c in COMPONENTS:
    print(f"  {c:<18}{100*F[c].notna().mean():5.1f}%")
F["n_components"] = F[COMPONENTS].notna().sum(axis=1)
print("\nnumber of components available per firm-year:")
print(F.n_components.value_counts().sort_index().to_string())
print(f"\n>= 2 components: {100*(F.n_components >= 2).mean():.1f}%   "
      f">= 3: {100*(F.n_components >= 3).mean():.1f}%")
print("\ncomponent correlations (should be positive but far from collinear --")
print("if any pair exceeds 0.9 they are measuring one thing, not two):")
display(F[COMPONENTS].corr().round(3))

## 7 · Build the index

In [ ]:
# =============================================================================
# CELL 7 - Build the index
#
# Each component is z-scored WITHIN YEAR, so the index measures a firm's
# position in that year's cross-section rather than a common time trend that
# would be confounded with the market itself. Equal weights, fixed in advance;
# PCA reported alongside as a robustness check, never as the headline.
#
# A firm needs at least two components. Averaging over available components
# rather than requiring all four is what keeps banks in the sample -- they have
# asset lightness and intangible share, and no gross margin.
# =============================================================================
MIN_COMPONENTS = 2
Z = F.copy()
for c in COMPONENTS:
    Z[f"z_{c}"] = Z.groupby("year")[c].transform(
        lambda s: (s - s.mean()) / s.std() if s.std() and s.notna().sum() > 5 else np.nan)
zc = [f"z_{c}" for c in COMPONENTS]
Z["bit_intensity"] = Z[zc].mean(axis=1)
Z.loc[Z.n_components < MIN_COMPONENTS, "bit_intensity"] = np.nan
# Re-standardise within year so the index is comparable across years despite
# firms contributing different numbers of components.
Z["bit_intensity"] = Z.groupby("year").bit_intensity.transform(
    lambda s: (s - s.mean()) / s.std())

ok = Z.bit_intensity.notna()
print(f"bit intensity computed for {int(ok.sum()):,} of {len(Z):,} firm-years "
      f"({100*ok.mean():.1f}%)")
print(f"covering {Z.loc[ok, 'ticker'].nunique()} firms")
print(f"\nmarket cap represented: {100*Z.loc[ok,'mktcap'].sum()/Z.mktcap.sum():.1f}%")

# --- PCA robustness -----------------------------------------------------------
sub = Z.loc[Z[zc].notna().all(axis=1), zc]
if len(sub) > 100:
    from numpy.linalg import eigh
    C = np.cov(sub.values.T)
    w, v = eigh(C)
    pc1 = v[:, np.argmax(w)]
    if pc1.sum() < 0:
        pc1 = -pc1
    print(f"\nPCA on the {len(sub):,} complete-case rows:")
    print(f"  first component explains {100*max(w)/w.sum():.1f}% of variance")
    print("  loadings: " + ", ".join(f"{c[2:]} {l:+.3f}" for c, l in zip(zc, pc1)))
    Z.loc[sub.index, "bit_intensity_pca"] = sub.values @ pc1
    r_pca = Z[["bit_intensity", "bit_intensity_pca"]].corr().iloc[0, 1]
    print(f"  corr(equal weights, PCA) = {r_pca:+.3f}")
    print("  A high correlation means the equal-weight choice is not doing the")
    print("  work; the components agree about which firms are bit-intensive.")

# --- sector-neutral variant --------------------------------------------------
# The dry run put Health Care at the top of the sector ranking, above Information
# Technology. That is not obviously wrong -- a pill costs almost nothing to
# reproduce and its value sits in the IP, which is the same economics chapter 6
# describes -- but it means the index could be reading "pharma" where the paper
# wants it to read "bits". The sector-neutral version z-scores WITHIN year AND
# sector, so it can only rank firms against their own industry. If H1 survives
# on both, the result is not a sector effect wearing a new name.
Z["bit_intensity_sn"] = Z.groupby(["year", "sector"]).bit_intensity.transform(
    lambda s: (s - s.mean()) / s.std() if s.notna().sum() > 5 else np.nan)
print(f"\nsector-neutral variant computed for "
      f"{int(Z.bit_intensity_sn.notna().sum()):,} firm-years")
print(f"corr(raw, sector-neutral) = "
      f"{Z[['bit_intensity','bit_intensity_sn']].corr().iloc[0,1]:+.3f}")

# Pooling raw market caps across years into one Hill estimate is wrong. The
# index total triples over the decade, so a pooled sample is a MIXTURE of
# differently-scaled distributions and is not Pareto even when every year is.
# The tail index is scale-invariant, so normalising each firm by its own year's
# index total removes the scale without touching the shape. Per-year estimates
# are unaffected; only the pooled ones need this.
Z["cap_share"] = Z.mktcap / Z.groupby("year").mktcap.transform("sum")

Z["bi_quartile"] = Z.groupby("year").bit_intensity.transform(
    lambda s: pd.qcut(s, 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
    if s.notna().sum() > 40 else np.nan)
print("\nfirms per bit-intensity quartile per year:")
display(Z.dropna(subset=["bi_quartile"]).pivot_table(
    index="year", columns="bi_quartile", values="ticker",
    aggfunc="count", observed=False).astype("Int64"))
savetable(Z[["year", "ticker", "roster_ticker", "sector", "mktcap", "recovered"]
           + COMPONENTS + CONTROLS
           + ["n_components", "bit_intensity", "bit_intensity_sn", "bi_quartile"]],
          "bit_intensity_panel")

## 8 · Does the index measure what it claims?

In [ ]:
# =============================================================================
# CELL 8 - Does the index measure what it claims to?
#
# Three checks, in order of how badly each could sink the paper.
#
# 1. Is it the GICS split renamed? Notebook 01's provisional two-component
#    version gave +0.194. If the full index comes back near 0.9 the paper is
#    relabelling a variable and there is nothing to write.
# 2. Does it rank firms the way the theory says it should? Software and
#    platforms high, utilities and heavy industry low. This is a face-validity
#    check and it is the one a referee will do first, by eye.
# 3. Is it stable within a firm over time? Bit intensity is a structural
#    property of a business model, so a firm whose index jumps between quartiles
#    year to year is measurement noise, not a signal.
# =============================================================================
DIGITAL_SECTORS = {"Information Technology", "Communication Services"}
DIGITAL_EXTRA = {"AMZN", "TSLA", "NFLX", "BKNG", "EBAY", "ABNB", "UBER", "DASH", "SHOP"}
Z["gics_digital"] = (Z.sector.isin(DIGITAL_SECTORS)
                     | Z.ticker.str.upper().isin(DIGITAL_EXTRA))

d = Z.dropna(subset=["bit_intensity"])
r_gics = float(d[["bit_intensity", "gics_digital"]].astype(float).corr().iloc[0, 1])
print("=" * 70)
print("CHECK 1 -- is bit intensity just the GICS digital label?")
print("=" * 70)
print(f"  corr = {r_gics:+.3f}   {'PASS' if abs(r_gics) < 0.90 else 'FAIL'}")
print(f"  notebook 01's two-component provisional value was +0.194")
print("\n  bit intensity by GICS group:")
display(d.groupby("gics_digital").bit_intensity.describe()
        [["count", "mean", "std", "25%", "50%", "75%"]].round(3))
overlap = d[d.gics_digital & (d.bit_intensity < d.bit_intensity.median())]
print(f"\n  {len(overlap):,} GICS-digital firm-years sit BELOW the median bit")
print("  intensity, and that overlap is the paper's whole argument: the sector")
print("  label and the economic property are not the same partition.")

print("\n" + "=" * 70)
print("CHECK 2 -- does it rank firms sensibly?")
print("=" * 70)
last = d[d.year == d.year.max()].sort_values("bit_intensity", ascending=False)
show = ["ticker", "sector", "bit_intensity"] + COMPONENTS
print("\ntop 20 by bit intensity, most recent year:")
display(last.head(20)[show].round(3))
print("\nbottom 15:")
display(last.tail(15)[show].round(3))
print("\nmean bit intensity by sector:")
display(d.groupby("sector").bit_intensity.agg(["count", "mean"]).round(3)
        .sort_values("mean", ascending=False))
print("\n  Read this critically. If utilities and REITs come out HIGH, the index")
print("  is picking up asset lightness in the accounting sense rather than in")
print("  the economic sense, and the component weights need rethinking.")

print("\n" + "=" * 70)
print("CHECK 3 -- is it stable within a firm?")
print("=" * 70)
piv = d.pivot_table(index="ticker", columns="year", values="bit_intensity")
within = piv.std(axis=1).median()
between = piv.mean(axis=1).std()
print(f"  median within-firm SD across years: {within:.3f}")
print(f"  between-firm SD of firm means:      {between:.3f}")
print(f"  ratio {within/between:.2f}   "
      f"{'stable -- structural, as intended' if within/between < 0.5 else 'NOISY -- investigate'}")
q = d.dropna(subset=["bi_quartile"]).pivot_table(
    index="ticker", columns="year", values="bi_quartile", aggfunc="first")
stay = q.apply(lambda r: r.dropna().nunique() <= 1, axis=1)
print(f"  {100*stay.mean():.0f}% of firms never change quartile across the decade")
savetable(pd.DataFrame({"metric": ["corr_with_gics", "within_firm_sd",
                                   "between_firm_sd", "share_never_change_quartile"],
                        "value": [r_gics, within, between, float(stay.mean())]}),
          "index_validation")

## 9 · H1 — tail index by bit intensity

In [ ]:
# =============================================================================
# CELL 9 - H1: does the tail get heavier as bit intensity rises?
#
#   H1  Near-zero marginal cost, not sector, drives tail position.
#       PREDICTION  the tail index falls monotonically across bit-intensity
#                   quartiles: Q4 (most bit-intensive) has the fattest tail.
#       FALSIFIED   if bit intensity adds nothing over the GICS split.
#
# The second half is what makes this a test rather than a description. A
# quartile gradient on its own could simply be the sector split in disguise, so
# the gradient is re-estimated WITHIN the GICS-digital group and within the
# non-digital group. If it survives inside both, bit intensity is carrying
# information the sector label does not have.
# =============================================================================
def alpha_by(panel, keys, min_n=MIN_N_ALPHA, min_k=GROUP_MIN_K, value="cap_share"):
    """Estimated on within-year normalised capitalisation. See cell 7: pooling
    raw caps across years mixes scales and destroys the Pareto shape."""
    rows = []
    for k, g in panel.groupby(keys, observed=True):
        x = g[value].dropna().values; x = x[x > 0]
        if len(x) < min_n:
            continue
        a, se, kk = hill_alpha(x, max(min_k, int(HILL_FRAC * len(x))))
        rec = dict(zip(keys if isinstance(keys, list) else [keys],
                       k if isinstance(k, tuple) else (k,)))
        rec.update({"n": len(x), "alpha_hill": a, "alpha_hill_se": se, "hill_k": kk,
                    "gini": gini(x), "top1_share": top_share(x, .01),
                    "mean_cap_share": float(x.mean())})
        rows.append(rec)
    return pd.DataFrame(rows)

dq = d.dropna(subset=["bi_quartile"]).copy()
by_q = alpha_by(dq, ["year", "bi_quartile"])
print("tail index by bit-intensity quartile and year:")
display(by_q.pivot(index="year", columns="bi_quartile", values="alpha_hill").round(3))
print("\npooled across years (each quartile's whole cross-section):")
pooled = alpha_by(dq, ["bi_quartile"], min_n=100)
display(pooled.round(4))
savetable(by_q, "alpha_by_bi_quartile")
savetable(pooled, "alpha_by_bi_quartile_pooled")

if len(pooled) == 4:
    order = ["Q1 low", "Q2", "Q3", "Q4 high"]
    a = pooled.set_index("bi_quartile").loc[order, "alpha_hill"]
    mono = bool((a.diff().dropna() < 0).all())
    print(f"\nQ1 {a.iloc[0]:.3f} -> Q4 {a.iloc[-1]:.3f}   "
          f"spread {a.iloc[0]-a.iloc[-1]:+.3f}")
    print(f"monotone decreasing across quartiles: {mono}")
    se = pooled.set_index("bi_quartile").loc[order, "alpha_hill_se"]
    z = (a.iloc[0] - a.iloc[-1]) / np.hypot(se.iloc[0], se.iloc[-1])
    p = 2 * (1 - st.norm.cdf(abs(z)))
    print(f"Q1 vs Q4:  z = {z:.2f}, p = {p:.4f}   "
          f"{'SEPARATE at 5%' if p < 0.05 else 'NOT separable'}")

# --- robustness: the same gradient on the sector-neutral index ---------------
sn = d.dropna(subset=["bit_intensity_sn"]).copy()
sn["sn_quartile"] = sn.groupby("year").bit_intensity_sn.transform(
    lambda s: pd.qcut(s, 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
    if s.notna().sum() > 40 else np.nan)
pooled_sn = alpha_by(sn.dropna(subset=["sn_quartile"]), ["sn_quartile"], min_n=100)
if len(pooled_sn):
    print("\nsame test on the SECTOR-NEUTRAL index (firms ranked only against")
    print("their own industry). If the gradient survives here it is not a")
    print("sector effect:")
    display(pooled_sn.round(4))
    savetable(pooled_sn, "alpha_by_bi_quartile_sector_neutral")

# --- the test that matters: does it survive inside the sector split? ---------
print("\n" + "=" * 70)
print("H1 -- bit intensity WITHIN the GICS partition")
print("=" * 70)
rows = []
for grp, sub in dq.groupby("gics_digital"):
    lab = "GICS digital" if grp else "GICS non-digital"
    sub = sub.copy()
    sub["local_half"] = sub.groupby("year").bit_intensity.transform(
        lambda s: pd.qcut(s, 2, labels=["low BI", "high BI"])
        if s.notna().sum() > 20 else np.nan)
    got = alpha_by(sub.dropna(subset=["local_half"]), ["local_half"], min_n=100)
    if len(got) == 2:
        g2 = got.set_index("local_half")
        lo, hi = g2.loc["low BI"], g2.loc["high BI"]
        z = (lo.alpha_hill - hi.alpha_hill) / np.hypot(lo.alpha_hill_se, hi.alpha_hill_se)
        p = 2 * (1 - st.norm.cdf(abs(z)))
        rows.append({"partition": lab, "n_low": int(lo.n), "n_high": int(hi.n),
                     "alpha_low_BI": round(lo.alpha_hill, 4),
                     "alpha_high_BI": round(hi.alpha_hill, 4),
                     "difference": round(hi.alpha_hill - lo.alpha_hill, 4),
                     "z": round(z, 2), "p": round(p, 4)})
within = pd.DataFrame(rows)
if len(within):
    display(within)
    print("\nH1 predicts a NEGATIVE difference in both rows -- higher bit")
    print("intensity, fatter tail, lower alpha -- and the row that matters is")
    print("GICS non-digital. A gradient there cannot be the sector label,")
    print("because every firm in that row carries the same label.")
    savetable(within, "h1_within_gics")

## 10 · H2 — platforms, holding bit intensity constant

In [ ]:
# =============================================================================
# CELL 10 - Platform coding completed, then H2
#
#   H2  Two-sided networks carry weight beyond asset-lightness.
#       PREDICTION  platform firms have a lower tail index than pipeline firms
#                   AT THE SAME BIT INTENSITY.
#       FALSIFIED   if the platform difference vanishes once bit intensity is
#                   controlled.
#
# The coding below extends notebook 01's pre-registered 27 with the firms from
# its review queue. It is still fixed BEFORE the tail indices in this cell are
# computed, ambiguous cases still go to pipeline, and every entry carries a
# reason. The three criteria are unchanged: two identifiable groups priced
# separately, cross-side benefit, and the firm sets rules rather than owning
# inventory.
# =============================================================================
PLATFORM_EXTRA = {
    "MCO":  False, "SPGI": False, "MSCI": False, "FDS": False,  # sold to one side
    "NVDA": False, "AVGO": False, "AMD": False, "MU": False,    # components
    "ORCL": False, "CRM": False, "NOW": False, "ADBE": False,   # licences
    "INTU": False, "WDAY": False, "PLTR": False, "IBM": False,
    "WMT":  False, "COST": False, "HD": False, "TGT": False,    # own inventory
    "JPM":  False, "BAC": False, "WFC": False, "GS": False, "MS": False,
    "TSLA": False, "DIS": False, "CMCSA": False, "NFLX": False,
    "BR":   True,   # Broadridge: intermediates issuers and investors
    "JKHY": True,   # Jack Henry: core processing between banks and networks
    "VRSN": False,  # registry operator, single-sided
    "GDDY": False,  # hosting and domains sold to one side
}
extra = pd.DataFrame([{"ticker": k, "platform": v,
                       "reason": "coded in notebook 03 from the review queue"}
                      for k, v in PLATFORM_EXTRA.items()])
coding = pd.concat([prereg[["ticker", "platform", "reason"]], extra], ignore_index=True)
coding = coding.drop_duplicates("ticker", keep="first")
coding["platform"] = coding.platform.astype(bool)

Z2 = d.merge(coding[["ticker", "platform"]], on="ticker", how="left")
# Uncoded firms default to pipeline, which works AGAINST H2 -- a missed platform
# dilutes the pipeline group rather than inflating the platform one.
n_uncoded = int(Z2.platform.isna().sum())
Z2["platform"] = Z2.platform.fillna(False)
n_plat_firms = int(Z2.loc[Z2.platform, "ticker"].nunique())
print(f"{len(coding)} firms coded; {n_plat_firms} platform firms in the panel")
print(f"{n_uncoded:,} firm-years default to pipeline (conservative against H2)")

print(f"\ngate: {n_plat_firms} platform firms  "
      f"{'PASS' if n_plat_firms >= 25 else 'FAIL -- H2 becomes a comparison of means'}")

print("\nbit intensity by business model -- H2 only means something if these")
print("overlap. If platforms are simply the bit-intensive firms, H2 and H1 are")
print("the same hypothesis:")
display(Z2.groupby("platform").bit_intensity.describe()
        [["count", "mean", "std", "25%", "50%", "75%"]].round(3))

pl = alpha_by(Z2, ["platform"], min_n=100)
print("\nunconditional:")
display(pl.round(4))

# --- H2 properly: platform vs pipeline WITHIN bit-intensity quartile ----------
print("\n" + "=" * 70)
print("H2 -- platform vs pipeline, HOLDING BIT INTENSITY CONSTANT")
print("=" * 70)
rows = []
for q, sub in Z2.dropna(subset=["bi_quartile"]).groupby("bi_quartile", observed=True):
    got = alpha_by(sub, ["platform"], min_n=60)
    if len(got) == 2:
        g2 = got.set_index("platform")
        z = ((g2.loc[False, "alpha_hill"] - g2.loc[True, "alpha_hill"])
             / np.hypot(g2.loc[False, "alpha_hill_se"], g2.loc[True, "alpha_hill_se"]))
        rows.append({"bi_quartile": str(q),
                     "n_pipeline": int(g2.loc[False, "n"]), "n_platform": int(g2.loc[True, "n"]),
                     "alpha_pipeline": round(g2.loc[False, "alpha_hill"], 4),
                     "alpha_platform": round(g2.loc[True, "alpha_hill"], 4),
                     "difference": round(g2.loc[True, "alpha_hill"]
                                         - g2.loc[False, "alpha_hill"], 4),
                     "z": round(float(z), 2),
                     "p": round(float(2 * (1 - st.norm.cdf(abs(z)))), 4)})
h2 = pd.DataFrame(rows)
if len(h2):
    display(h2)
    print("\nH2 predicts a NEGATIVE difference in every quartile. A difference")
    print("that appears unconditionally and vanishes here is bit intensity")
    print("wearing a platform label, and H2 is not supported.")
    savetable(h2, "h2_platform_within_bi")
savetable(coding, "platform_coding_final")
savetable(Z2[["year", "ticker", "sector", "mktcap", "bit_intensity",
              "bi_quartile", "gics_digital", "platform"]], "firm_classification_final")

## 11 · Figures and download

In [ ]:
# =============================================================================
# CELL 11 - Figures, manifest, zip
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
INK, ACC, GREY = "#1f3b57", "#c8702a", "#7a7a7a"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure {name}")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
ax = axes[0]
ax.plot(covdf.year, covdf.pct_published, "o-", color=ACC, lw=2, ms=4, label="published")
ax.plot(covdf.year, covdf.pct_final, "o-", color=INK, lw=2, ms=4, label="repaired")
ax.set_xlabel("year"); ax.set_ylabel("% of roster in the panel")
ax.set_title("Coverage after the rename repair"); ax.legend(fontsize=8)

ax = axes[1]
piv_q = by_q.pivot(index="year", columns="bi_quartile", values="alpha_hill")
cm = plt.cm.viridis(np.linspace(0.15, 0.85, piv_q.shape[1]))
for c, col in zip(cm, piv_q.columns):
    ax.plot(piv_q.index, piv_q[col], "o-", color=c, lw=1.8, ms=3.5, label=str(col))
ax.axhline(1.0, color=GREY, lw=1, ls=":")
ax.set_xlabel("year"); ax.set_ylabel("Hill alpha")
ax.set_title("Tail index by bit-intensity quartile"); ax.legend(fontsize=7.5)

ax = axes[2]
dd = d.dropna(subset=["bit_intensity"])
for lab, sub, c in [("GICS digital", dd[dd.gics_digital], INK),
                    ("GICS non-digital", dd[~dd.gics_digital], ACC)]:
    ax.hist(sub.bit_intensity, bins=40, alpha=0.5, color=c, label=lab, density=True)
ax.set_xlabel("bit intensity (within-year z)"); ax.set_ylabel("density")
ax.set_title("The two partitions overlap"); ax.legend(fontsize=8)
fig.suptitle("Notebook 03: the repaired panel and the new measure",
             fontsize=10, color=GREY, y=1.02)
savefig(fig, "B1_repair_and_index")

files = sorted(p for p in OUTD.rglob("*") if p.is_file())
rows = []
for p in files:
    row = {"file": str(p.relative_to(OUTD)), "kb": round(p.stat().st_size / 1024, 1)}
    if p.suffix == ".parquet":
        try:
            row["rows"] = len(pd.read_parquet(p))
        except Exception:
            row["rows"] = None
    rows.append(row)
manifest = pd.DataFrame(rows); manifest.to_csv(OUTD / "MANIFEST.csv", index=False)
display(manifest)

summary = {"run": RUN_TAG, "when": datetime.datetime.now().isoformat(timespec="seconds"),
           "aliases": ALIASES, "panel_rows": int(len(rep)),
           "recovered": int(rep.recovered.sum()),
           "editor_table": editor.to_dict("records"),
           "bit_intensity_coverage": float(ok.mean()),
           "corr_with_gics": float(r_gics),
           "platform_firms": int(n_plat_firms),
           "tables": saved_tables}
(OUTD / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))

print("\n" + "=" * 70)
print("SEND ME")
print("=" * 70)
print("  editor_robustness_table.csv   whether the JRFM claims survive")
print("  index_validation.csv          is bit intensity a real new variable")
print("  alpha_by_bi_quartile_pooled.csv + h1_within_gics.csv    H1")
print("  h2_platform_within_bi.csv     H2")
print("  and the top/bottom-20 printout from cell 8, which is the check no")
print("  statistic can do for me -- whether the ranking looks right to someone")
print("  who knows these companies.")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(str(BASE / f"economy_of_bits_{RUN_TAG}_{stamp}"),
                              "zip", root_dir=OUTD.parent, base_dir=OUTD.name)
print(f"\nzipped -> {archive}  ({pathlib.Path(archive).stat().st_size/1024:.0f} KB)")
try:
    from google.colab import files as colab_files
    colab_files.download(archive); print("Download started.")
except ImportError:
    print("Not on Colab. Copy the archive from the path above.")